### Task 1: Parquet Internals and Metadata Inspection
**Step 1**: Create a dataset with 500,000 rows using the following schema (you will reuse this dataset throughout the lab). Use np.random.seed(42) for reproducibility.

**Step 2**: Save the dataset as a Parquet file. Then use pyarrow.parquet.ParquetFile to inspect:

The number of row groups
The number of rows and columns
The schema (column names and types)
For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)

**Step 3**: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

**Step 4**: Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.

In [1]:
#Step1
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500_000

cities = ["Berlin", "Tokyo", "New York", "London", "Paris",
          "Baku", "Dubai", "Toronto", "Sydney", "Rome"]

df = pd.DataFrame({
    "user_id": np.arange(1, n+1),
    "city": np.random.choice(cities, n),
    "score": np.random.uniform(0, 100, n),
    "active": np.random.choice([True, False], n),
    "signup_date": pd.to_datetime("today") - pd.to_timedelta(
        np.random.randint(0, 365*3, n), unit="d"
    ),
    "age": np.random.randint(18, 80, n),
    "sessions": np.random.randint(0, 500, n),
    "revenue": np.random.uniform(0, 1000, n)
})

df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Dubai,43.689490,False,2026-01-31 21:55:12.154161,71,14,167.891338
1,2,London,97.403462,True,2025-06-01 21:55:12.154161,39,399,546.244310
2,3,Toronto,66.148238,False,2025-11-07 21:55:12.154161,23,283,544.570185
3,4,Paris,21.993544,False,2025-05-21 21:55:12.154161,20,205,421.757634
4,5,Dubai,91.719953,True,2023-04-01 21:55:12.154161,67,113,603.913530


In [2]:
#Step 2
import pyarrow as pa
import pyarrow.parquet as pq
pq.write_table(pa.Table.from_pandas(df),"data.parquet")

In [3]:
pf=pq.ParquetFile("data.parquet")
print("Row groups:", pf.num_row_groups)
print("Rows:",pf.metadata.num_rows)
print("Columns:",pf.metadata.num_columns)

Row groups: 1
Rows: 500000
Columns: 8


In [4]:
print("\nSchema:")
print(pf.schema)


Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}



In [5]:
#first row group
rg= pf.metadata.row_group(0)

In [6]:
for i in range(rg.num_columns):
    col = rg.column(i)
    stats = col.statistics

    print(f"\nColumn: {col.path_in_schema}")
    print("Physical type:", col.physical_type)
    print("Total compressed size:", col.total_compressed_size)

    if stats:
        print("Min:", stats.min)
        print("Max:", stats.max)
        print("Null count:", stats.null_count)


Column: user_id
Physical type: INT64
Total compressed size: 2273849
Min: 1
Max: 500000
Null count: 0

Column: city
Physical type: BYTE_ARRAY
Total compressed size: 252561
Min: Baku
Max: Toronto
Null count: 0

Column: score
Physical type: DOUBLE
Total compressed size: 4272959
Min: 5.188445665327279e-05
Max: 99.99983148609545
Null count: 0

Column: active
Physical type: BOOLEAN
Total compressed size: 63800
Min: False
Max: True
Null count: 0

Column: signup_date
Physical type: INT64
Total compressed size: 699133
Min: 2023-03-06 21:55:12.154161
Max: 2026-03-04 21:55:12.154161
Null count: 0

Column: age
Physical type: INT32
Total compressed size: 377943
Min: 18
Max: 79
Null count: 0

Column: sessions
Physical type: INT32
Total compressed size: 567222
Min: 0
Max: 499
Null count: 0

Column: revenue
Physical type: DOUBLE
Total compressed size: 4272959
Min: 0.0009958058022618843
Max: 999.9978869129644
Null count: 0


In [7]:
#Step 3
df.to_csv("data.csv",index=False)

In [8]:
import os

In [9]:
csv_size=os.path.getsize("data.csv")/1024
parquet_size=os.path.getsize("data.parquet")/1024

In [10]:
print(f"Parquet size: {parquet_size:.2f} KB")
print(f"CSV size: {csv_size:.2f} KB")

print(f"Compression ratio: {csv_size / parquet_size:.2f}x")

Parquet size: 12485.54 KB
CSV size: 43946.59 KB
Compression ratio: 3.52x


Parquet metadata provides detailed structural and statistical information that CSV files do not contain.

Parquet stores:
- Schema (column names and data types)
- Row group structure
- Column-level metadata (min, max, null count)
- Compression information

CSV, on the other hand:
- Stores only raw text
- Has no schema or metadata
- Requires full scanning to understand data

Why this matters for performance:

Parquet enables:
- Column pruning (reading only needed columns)
- Predicate pushdown (skipping irrelevant data)
- Efficient compression and encoding

CSV requires:
- Full file scan
- Parsing text into types
- No skipping of irrelevant data

As a result, Parquet is significantly faster and more efficient for analytical workloads.

### Task 2: Column Projection and Selective Reads
Step 1: Read the full Parquet file from Task 1 and time it.

Step 2: Read only 2 columns from the same Parquet file and time it.

Step 3: Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).

Step 4: Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

In [11]:
#Step 1
import time
start=time.time()
df_full=pd.read_parquet("data.parquet")
print("Parquet full read :", time.time()-start)

Parquet full read : 0.474470853805542


In [12]:
#Step 2
# 2 columns Parquet
start=time.time()
df_cols=pd.read_parquet("data.parquet",columns=["city", "score"])
print("Parquet 2 columns read: ",time.time()-start)

Parquet 2 columns read:  0.0836637020111084


In [13]:
#Step 3
start=time.time()
df_csv=pd.read_csv("data.csv")[["city", "score"]]
print("CSV 2 columns :", time.time()-start)

CSV 2 columns : 0.9130029678344727


Step 4
Parquet selective reads are faster because of its columnar storage format.

In Parquet:
- Each column is stored separately (column chunks)
- Reading 2 columns means only those columns are loaded

In CSV:
- Data is stored row-wise
- Entire file must be read and parsed before selecting columns

Thus:
Parquet avoids unnecessary I/O, making it significantly faster.


### Task 3: Predicate Pushdown in Practice
Step 1: Using PyArrow, read the Parquet file with a filter (e.g., age > 50). Time the read.

Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.

Step 3: Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.

Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:

import duckdb
result = duckdb.sql("SELECT * FROM 'your_file.parquet' WHERE age > 50").df()
Time it and compare with both PyArrow and pandas approaches.

In [14]:
#Step1
import pyarrow.parquet as pq
import time

start = time.time()

table = pq.read_table(
    "data.parquet",
    filters=[("age", ">", 50)]
)

# Convert to pandas DataFrame
df_arrow = table.to_pandas()

end = time.time()
time_arrow = end - start

print("PyArrow filter time:", time_arrow, "seconds")
print("Rows returned (PyArrow):", len(df_arrow))

PyArrow filter time: 0.0959937572479248 seconds
Rows returned (PyArrow): 233565


In [15]:
import pandas as pd
import time

start = time.time()

df_full = pd.read_parquet("data.parquet")

df_filtered = df_full[df_full["age"] > 50]

end = time.time()
time_pandas = end - start

print("Pandas full load + filter time:", time_pandas, "seconds")
print("Rows returned (Pandas):", len(df_filtered))

Pandas full load + filter time: 0.1194460391998291 seconds
Rows returned (Pandas): 233565


In [16]:
rows_arrow = len(df_arrow)
rows_pandas = len(df_filtered)

print(f"Rows PyArrow: {rows_arrow}")
print(f"Rows Pandas: {rows_pandas}")
print("Results identical:", rows_arrow == rows_pandas)

print("\nExecution times (seconds):")
print(f"PyArrow (predicate pushdown): {time_arrow:.2f}")
print(f"Pandas (post-filter): {time_pandas:.2f}")

Rows PyArrow: 233565
Rows Pandas: 233565
Results identical: True

Execution times (seconds):
PyArrow (predicate pushdown): 0.10
Pandas (post-filter): 0.12


**Predicate Pushdown Explanation**

Predicate pushdown is an optimization where the filter condition (e.g., `age > 50`) is applied **while reading the data**, instead of after the full dataset is loaded.

- **PyArrow**: Only reads relevant row groups in the Parquet file. Parquet stores column-level statistics (min, max, null counts) per row group, allowing irrelevant row groups to be skipped.
- **Pandas**: Reads the entire dataset into memory first, then applies the filter. More I/O, more memory usage, and slower execution.

**Why PyArrow is faster**:
- Reduces the amount of data read from disk
- Avoids unnecessary parsing and memory allocation
- Speeds up analytical queries

**Observation**:
- Number of rows returned is identical across both methods
- PyArrow execution time is much lower than pandas because of predicate pushdown

In [17]:
#Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:
import duckdb
start=time.time()
result=duckdb.sql(
    "select * from 'data.parquet' where age >50"
).df()

end=time.time()

time_duckdb=end-start

print("DuckDB filter time:", time_duckdb, "seconds")
print("Rows returned (DuckDB):", len(result))


DuckDB filter time: 0.12247014045715332 seconds
Rows returned (DuckDB): 233565


**DuckDB Filter**

DuckDB can query Parquet files directly using SQL. It supports predicate pushdown:

- Only relevant row groups are read
- Disk I/O is minimized
- Execution is faster than pandas full scan

In [18]:

print("\nExecution times (seconds):")
print(f"PyArrow (predicate pushdown): {time_arrow:.2f}")
print(f"Pandas (full load + filter): {time_pandas:.2f}")
print(f"DuckDB (SQL filter + pushdown): {time_duckdb:.2f}")




Execution times (seconds):
PyArrow (predicate pushdown): 0.10
Pandas (full load + filter): 0.12
DuckDB (SQL filter + pushdown): 0.12


### Task 4: pandas vs DuckDB — Identical Queries
Run the same five analytical queries in both pandas and DuckDB on the dataset from Task 1. For each query, record the code and the execution time.

**Query 1**: Count records per city.

**Query 2**: Average score by city, ordered by average score descending.

**Query 3**: For each city, find the percentage of active users whose score is above 75.

**Query 4**: Find the top 10 users by score for each city (window function / rank).

**Query 5**: Compute a running total of scores ordered by user_id, partitioned by city.

For DuckDB, write the query in SQL using duckdb.sql(). For pandas, use native pandas operations (groupby, transform, rank, etc.).

In [19]:
#query 1

# --- Pandas ---
start = time.time()
pandas_q1 = df.groupby("city").size().reset_index(name="count")
time_pandas_q1 = time.time() - start

#-----duckdb----
start = time.time()
duckdb_q1 = duckdb.sql("""
SELECT city, COUNT(*) AS count
FROM df
GROUP BY city
""").df()
time_duckdb_q1 = time.time() - start

print(f"Query 1 times: pandas={time_pandas_q1:.2f}s, duckdb={time_duckdb_q1:.2f}s")

Query 1 times: pandas=0.05s, duckdb=0.03s


In [20]:
#query 2

# --- Pandas ---
start=time.time()
pandas_q2=df.groupby("city")["score"].mean().reset_index().sort_values("score", ascending=False)
time_pandas_q2=time.time()-start


# ---- Duckdb---
start=time.time()
duckdb_q2=duckdb.sql(
    """
    select city , avg(score) 
    from df
    group by city
    order by avg(score) desc """
).df()

time_duckdb_q2=time.time()-start

print(f"Query 2 times: pandas={time_pandas_q2:.2f}s, duckdb={time_duckdb_q2:.2f}s")

Query 2 times: pandas=0.06s, duckdb=0.02s


In [21]:
#query 3 For each city, find the percentage of active users whose score is above 75.

# --- Pandas ----
start=time.time()
pandas_q3=df.groupby("city")[["active", "score"]].apply(  lambda x: ((x["active"]) & (x["score"] > 75)).mean() * 100
).reset_index(name="pct_active_score75")
time_pandas_q3 = time.time() - start


# ----Duckdb---
start=time.time()
duckdb_q3=duckdb.sql(
    """
    SELECT city,
       AVG(CASE WHEN active AND score > 75 THEN 1 ELSE 0 END)*100.0 AS pct_active_score75
    FROM df
    GROUP BY city
"""
).df()
time_duckdb_q3=time.time()-start

print(f"Query 3 times : pandas={time_pandas_q3:.2f}s, duckdb={time_duckdb_q3:.2f}.s")

Query 3 times : pandas=0.09s, duckdb=0.03.s


In [22]:
# Query 4 : Find the top 10 users by score for each city (window function / rank).
# --- Pandas ----
start = time.time()
df["rank"] = df.groupby("city")["score"].rank(method="first", ascending=False)
pandas_q4 = df[df["rank"] <= 10].sort_values(["city", "rank"])
time_pandas_q4 = time.time() - start


# --- Duckdb ----
start=time.time()
duckdb_q4=duckdb.sql(
    """
    select * from (
    select city,
    dense_rank() over (partition by city order by score desc ) as rn
    from df)
    where rn<11
    order by city,rn
    """
).df()
time_duckdb_q4=time.time()-start

print(f"Query 4 times : pandas={time_pandas_q4:.2f}s, duckdb={time_duckdb_q4:.2f}.s")

Query 4 times : pandas=0.45s, duckdb=0.14.s


In [26]:
#Query 5: Compute a running total of scores ordered by user_id, partitioned by city.
# --- Pandas ----
start = time.time()
pandas_q5=df.sort_values("user_id").copy()
pandas_q5["running_total"] = pandas_q5.groupby("city")["score"].cumsum()
time_pandas_q5 = time.time() - start


# --- Duckdb -----
start=time.time()
duckdb_q5=duckdb.sql(
    """
    select *,
    sum(score) over (partition by city order by score ) as running_total
    from df
    """
).df()
time_duckdb_q5=time.time()-start

print(f"Query 4 times : pandas={time_pandas_q5:.2f}s, duckdb={time_duckdb_q5:.2f}.s")


Query 4 times : pandas=0.16s, duckdb=0.40.s


### Task 5: Arrow as the Bridge
Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().

Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.

Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.

Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.

Step 5: Demonstrate the pandas dtype_backend="pyarrow" option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.

Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?

In [27]:
#step1
df_sample=df.copy()
arrow_table=pa.Table.from_pandas(df_sample)
print(arrow_table.schema)

user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
rank: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1290


In [31]:
#step2
print("\nPandas dytpes: ",df_sample.dtypes)
print("-"*50)
print("\nArrow schema: ", arrow_table.schema)


Pandas dytpes:  user_id                 int64
city                   object
score                 float64
active                   bool
signup_date    datetime64[ns]
age                     int32
sessions                int32
revenue               float64
rank                  float64
dtype: object
--------------------------------------------------

Arrow schema:  user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
rank: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1290


In [35]:
#Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.
pq.write_table(arrow_table,"arrow_parquet.parquet")

arrow_table2=pq.read_table("arrow_parquet.parquet")
print(arrow_table2.schema)

user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
rank: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1290


In [38]:
#Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.
df_from_arrow=arrow_table2.to_pandas()
print("Data matches:", df_sample.equals(df_from_arrow))

Data matches: True


In [39]:
#Step 5: Demonstrate the pandas dtype_backend="pyarrow" option 
#by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.
df_arrow_backed=pd.read_parquet("arrow_parquet.parquet",dtype_backend="pyarrow")
print(df_arrow_backed.dtypes)

user_id                int64[pyarrow]
city                  string[pyarrow]
score                 double[pyarrow]
active                  bool[pyarrow]
signup_date    timestamp[ns][pyarrow]
age                    int32[pyarrow]
sessions               int32[pyarrow]
revenue               double[pyarrow]
rank                  double[pyarrow]
dtype: object


#### Step 6
**Role of Arrow in the Modern Analytics Stack**

1. **Arrow as bridge:**
   - **Disk (Parquet)** → columnar storage
   - **Arrow Table** → in-memory columnar format
   - **pandas** → analysis
   - **DuckDB** → SQL analytics

2. **Benefits:**
   - Zero-copy conversion between pandas and Arrow
   - Fast reading/writing of Parquet files
   - Efficient memory usage for columnar analytics
   - Enables interoperability between different analytics tools

3. **Example workflow:**
   - Read Parquet → Arrow Table → pandas DataFrame → DuckDB SQL query
   - Arrow ensures type consistency and fast in-memory operations